# 01 · Data & quality
**Project:** freddie-pd-build &nbsp;·&nbsp; **Type:** analysis

| | |
|---|---|
| **Purpose** | Extract the data the question needs and make sure it can be trusted. |
| **Inputs** | `sql/01_extract.sql` |
| **Outputs** | `data/interim/01_base` |
| **Assumptions** | |
| **Status** | 🟡 in progress |

## Setup

In [ ]:
import sys
import numpy as np
import pandas as pd
from er_toolkit import brand, contracts, project as proj

cfg = proj.config()                       # ../config.yml
P = proj.paths()                          # P.raw, P.interim, P.sql, P.tables, P.figures …
sys.path.insert(0, str(P.root / "src"))   # project helpers: from <pkg> import …
b = brand.use(cfg.get("brand"))
SNAPSHOT = pd.Timestamp(cfg["snapshot_date"])
STAGE = "01_data"

## 1. Extract
SQL lives in `sql/01_extract.sql` (start from the toolkit's SQL patterns). Parameters come from `config.yml`.

In [ ]:
import duckdb
from er_toolkit.io.sql import run_sql

con = duckdb.connect(str(P.root / cfg["data"]["database"]), read_only=True)
df = run_sql(P.sql / "01_extract.sql", con, snapshot_date=str(SNAPSHOT.date()))
df.shape

## 2. Data quality
Types, missingness, duplicates on the key. Note anything odd in the Log.

In [ ]:
KEY = cfg["data"].get("key", "loan_id")
dq = pd.DataFrame({"dtype": df.dtypes.astype(str), "missing": df.isna().mean(),
                   "n_unique": df.nunique()}).sort_values("missing", ascending=False)
print(f"{len(df):,} rows · {df[KEY].duplicated().sum():,} duplicate {KEY}")
dq.style.format({"missing": "{:.1%}"})

## 3. Exclusions
Every row you drop, and why — in order.

In [ ]:
# One line per exclusion rule, in order — this table goes straight into the documentation.
rules = {
    # "Not yet matured (< performance window)": df["orig_date"] > SNAPSHOT - pd.DateOffset(months=12),
    # "Fraud / test accounts": df["fraud_flag"] == 1,
}
waterfall, keep = [("Extract", len(df))], pd.Series(True, index=df.index)
for name, mask in rules.items():
    keep &= ~mask
    waterfall.append((name, int(keep.sum())))
df = df[keep].copy()
pd.DataFrame(waterfall, columns=["step", "rows_remaining"])

## Handoff
What the next stage / the dashboard reads. Tables written with `write_table` are checked against their contract and logged in `outputs/manifest.json`.

In [ ]:
proj.write_interim(df, "01_base")

## Log
| Date | Decision / finding / open question |
|---|---|
| 2026-10-16 | Notebook created |